In [ ]:
%run ./utils_common

In [ ]:
logger = setup_logger("TotalPoolSpendsReporter")

In [ ]:
dbutils.widgets.text("catalog", "", "CATALOG")
dbutils.widgets.text("schema", "", "SCHEMA")
dbutils.widgets.text("overlap_days", "3", "Overlap days (min 2)")

In [ ]:
# =======================================================
# Total Pool Spends Client
# =======================================================
# Sibling of TotalJobSpendsClient and TotalAllPurposeSpendsClient.
# Differences:
#   * source DBU table: dbspend360_pool_dbu_cost
#                       (keyed (instance_pool_id, cluster_id, usage_date))
#   * target table:     dbspend360_total_pool_spends
#   * cloud-cost join (CP6 / plan §4.4): pool EC2/EBS is sourced from
#     the dedicated dbspend360_pool_cloud_cost_explorer table (idle/warm
#     pool capacity is tagged DatabricksInstancePoolId, NOT ClusterId, so
#     it never reaches dbspend360_cloud_cost_explorer). The join is DRIVEN
#     FROM the cloud table, not the rollup: idle pool-days emit zero
#     system.billing.usage rows, so a LEFT join onto the rollup would
#     silently drop the idle EC2 cost we most want. Pool VM cost is
#     pool-level (not attributable to an attached cluster), so it lands on
#     the synthetic __pool_overhead__ row (synthesized for idle-only
#     pool-days); per-cluster rows keep cloud_cost = NULL so the UI renders
#     "—" with the pool-level note (plan §5). The CP5 ClusterId-netting
#     guard already removed any pool cost that ALSO carried ClusterId, so
#     this is disjoint from the cluster/all-purpose/pipeline tabs.
#   * reconciliation invariant (plan §4.6): SUM(cloud_cost) per
#     (instance_pool_id, usage_date) must equal the pool explorer within
#     0.01 USD; asserted on the pre-MERGE source. A post-write monitor
#     mirrors the explorer's: cloud ~0 while DBU non-zero raises an alarm.
#   * adds an SCD-collapse of system.compute.instance_pools so the
#     rollup denormalizes pool metadata (pool_name, node_type,
#     min_idle_instances, max_capacity,
#     idle_instance_autotermination_minutes) and the per-pool
#     delete_time → pool_deleted_at column for the §3.5 three-state
#     snapshot badge (active / deleted-visible / snapshot-missing).
#   * pool_snapshot_missing is computed BEFORE the COALESCE fallback
#     on pool_name so it captures the underlying snapshot state, not
#     the post-fallback state.
#   * Creator info is intentionally NOT denormalized.
#     system.compute.instance_pools.tags is documented as user-defined
#     tags only (excludes default tags), so the auto-applied
#     DatabricksInstancePoolCreatorId tag is not visible from the
#     system table. Creator GUID is resolved per-request in the pool
#     details modal via the REST API (§4.1, CP6).
#   * MERGE key: (instance_pool_id, cluster_id, usage_date)
class TotalPoolSpendsClient:

    TABLE_NAME = "dbspend360_total_pool_spends"

    def __init__(
        self,
        audit_table: str,
        pool_cloud_cost_table: str,
        databricks_cost_table: str,
        target_table: str,
        error_log_table: str,
        overlap_days: int,
        logger=None,
    ):
        self.audit_table = audit_table
        self.pool_cloud_cost_table = pool_cloud_cost_table
        self.databricks_cost_table = databricks_cost_table
        self.target_table = target_table
        self.error_log_table = error_log_table
        self.overlap_days = overlap_days
        self.logger = logger or logging.getLogger("TotalPoolSpendsClient")

    def _load_pool_snapshot(self):
        # SCD-collapse system.compute.instance_pools to one row per
        # instance_pool_id carrying the most-recent metadata.
        # max_by(col, change_time) mirrors the pattern used in
        # dbspend360_all_purpose_dbu_cost_app.ipynb; the QUALIFY
        # ROW_NUMBER() OVER (... ORDER BY change_time DESC) = 1
        # alternative is holistically safer on tied change_time values
        # but breaks consistency with the existing pipeline (plan §5.5).
        #
        # Per-field notes (verified against the published
        # system.compute.instance_pools schema):
        #   * The actual column is `node_type`, NOT `node_type_id`.
        #   * `delete_time` is non-null iff the pool was deleted; the
        #     most-recent SCD row carries the delete timestamp. Carry
        #     it through as `pool_deleted_at` so the UI can render the
        #     "Deleted YYYY-MM-DD" badge from plan §3.5.
        #   * `tags['DatabricksInstancePoolCreatorId']` is NOT read
        #     here because the system table's `tags` column excludes
        #     default tags (it is documented "User-defined tags for the
        #     instance pool (does not include default tags)"), so it
        #     would return NULL on every row. Creator info is resolved
        #     per-request in the modal path via the REST API (CP6).
        return spark.sql("""
            SELECT instance_pool_id,
                   max_by(instance_pool_name,                    change_time) AS pool_name,
                   max_by(node_type,                             change_time) AS node_type,
                   max_by(min_idle_instances,                    change_time) AS min_idle_instances,
                   max_by(max_capacity,                          change_time) AS max_capacity,
                   max_by(idle_instance_autotermination_minutes, change_time)
                                                                              AS idle_instance_autotermination_minutes,
                   max_by(delete_time,                           change_time) AS pool_deleted_at
            FROM system.compute.instance_pools
            GROUP BY instance_pool_id
        """)

    def _assert_reconciliation(self, final_df, cloud_df, start_dt, end_dt):
        # Plan §4.6 invariant: for every (instance_pool_id, usage_date),
        # SUM(cloud_cost) in the rollup must equal the pool explorer's
        # cloud_cost within 0.01 USD. Asserted on the pre-MERGE source (the
        # MERGE writes these values verbatim for the window). Summed across
        # currency on both sides because the rollup MERGE key has no currency
        # and pool cloud is single-currency in practice (plan §4.4); a future
        # multi-currency pool-day is flagged separately in the build path.
        # A FULL OUTER join catches BOTH dropped explorer cost (idle cloud
        # that failed to synthesize) and phantom written cost (cloud with no
        # explorer row) — both are hard errors.
        written = (
            final_df
            .groupBy("instance_pool_id", "usage_date")
            .agg(F.sum(F.coalesce(F.col("cloud_cost"), F.lit(0.0))).alias("written_cloud_cost"))
        )
        explorer = (
            cloud_df
            .groupBy("instance_pool_id", "usage_date")
            .agg(F.sum(F.coalesce(F.col("cloud_cost"), F.lit(0.0))).alias("explorer_cloud_cost"))
        )
        mismatch_df = (
            written.join(explorer, on=["instance_pool_id", "usage_date"], how="full_outer")
            .withColumn("written_cloud_cost", F.coalesce(F.col("written_cloud_cost"), F.lit(0.0)))
            .withColumn("explorer_cloud_cost", F.coalesce(F.col("explorer_cloud_cost"), F.lit(0.0)))
            .withColumn("diff", F.abs(F.col("written_cloud_cost") - F.col("explorer_cloud_cost")))
            .filter(F.col("diff") > 0.01)
        )

        mismatch_count = mismatch_df.count()
        if mismatch_count == 0:
            self.logger.info(
                f"Reconciliation OK: rollup cloud_cost matches "
                f"{self.pool_cloud_cost_table} ± 0.01 USD per (pool, day) "
                f"for {start_dt} → {end_dt}"
            )
            return

        err_df = (
            mismatch_df
            .select(
                F.lit("RECONCILIATION").alias("source_system"),
                F.lit("POOL_CLOUD_COST_MISMATCH").alias("error_type"),
                F.lit(None).cast("string").alias("cluster_id"),
                F.lit(None).cast("string").alias("job_id"),
                F.lit(None).cast("string").alias("run_id"),
                F.col("usage_date"),
                F.lit(None).cast("string").alias("currency"),
                F.concat(
                    F.lit("pool cloud_cost mismatch: pool="),
                    F.col("instance_pool_id"),
                    F.lit(", written="),
                    F.col("written_cloud_cost").cast("string"),
                    F.lit(", explorer="),
                    F.col("explorer_cloud_cost").cast("string"),
                    F.lit(", diff="),
                    F.col("diff").cast("string"),
                ).alias("error_detail"),
                F.to_json(F.struct(
                    F.col("instance_pool_id"), F.col("usage_date"),
                    F.col("written_cloud_cost"), F.col("explorer_cloud_cost"),
                    F.col("diff"),
                )).alias("raw_record"),
            )
            .withColumn("created_at", F.lit(datetime.now(timezone.utc)))
        )
        try:
            _safe_append(err_df, self.error_log_table)
        except Exception as e:
            self.logger.exception(
                f"Failed to write pool reconciliation mismatches to error log: {e}"
            )

        sample = mismatch_df.limit(5).collect()
        sample_str = "; ".join(
            f"(pool={r['instance_pool_id']}, date={r['usage_date']}, "
            f"written={r['written_cloud_cost']}, explorer={r['explorer_cloud_cost']}, "
            f"diff={r['diff']:.4f})"
            for r in sample
        )
        raise DataQualityError(
            f"Pool reconciliation invariant violated: {mismatch_count} "
            f"(instance_pool_id, usage_date) keys differ from "
            f"{self.pool_cloud_cost_table} by > 0.01 USD. Sample: {sample_str}"
        )

    def _monitor_post_write(self, final_df, start_dt, end_dt):
        # Plan §4.6 advisory monitor (mirrors the explorer's
        # _monitor_pool_post_write). If the window's pool cloud_cost collapses
        # to ~0 while pool DBU is non-zero, surface a non-silent alarm
        # (suspected DatabricksInstancePoolId tag lapse or an empty pool Cost
        # Explorer table). Advisory only — never fails the run; the hard gate
        # is _assert_reconciliation above.
        agg = final_df.agg(
            F.sum(F.coalesce(F.col("cloud_cost"), F.lit(0.0))).alias("cloud"),
            F.sum(F.coalesce(F.col("databricks_cost"), F.lit(0.0))).alias("dbu"),
        ).collect()[0]
        total_cloud = float(agg["cloud"] or 0.0)
        total_dbu = float(agg["dbu"] or 0.0)

        if total_cloud < 0.01 and total_dbu > 0.0:
            alert = (
                f"Pool rollup cloud_cost for {start_dt} → {end_dt} is "
                f"{total_cloud:.4f} (~0) while pool DBU is {total_dbu:.2f}; "
                f"suspected DatabricksInstancePoolId tag lapse or empty pool "
                f"Cost Explorer table."
            )
            self.logger.error(f"[POOL ROLLUP MONITOR ALARM] {alert}")
            try:
                write_error_log_entries(
                    [alert], "POOL_ROLLUP", "POOL_COST_MONITOR_ALARM", self.error_log_table,
                )
            except Exception as e:
                self.logger.error(
                    f"Failed to persist pool rollup monitor alarm to error_log: {e}"
                )
        else:
            self.logger.info(
                f"Pool rollup post-write monitor OK for {start_dt} → {end_dt}: "
                f"cloud_cost={total_cloud:.4f}, dbu={total_dbu:.2f}."
            )

    def build_total_pool_spends(self):
        start_dt = end_dt = datetime.now(timezone.utc).date()
        try:
            start_dt, end_dt = get_date_window(self.audit_table, self.TABLE_NAME, self.overlap_days)

            valid, msg = validate_date_window(start_dt, end_dt)
            if not valid:
                raise DataQualityError(msg)

            self.logger.info(
                f"Building dbspend360_total_pool_spends for {start_dt} → {end_dt}"
            )

            dbu_df = (
                spark.table(self.databricks_cost_table)
                    .alias("dbu")
                    .filter(
                        (F.col("usage_date") >= F.lit(start_dt)) &
                        (F.col("usage_date") <= F.lit(end_dt))
                    )
            )

            # CP6 cloud-cost source (plan §4.4). Pool EC2/EBS lives in its own
            # explorer table keyed (instance_pool_id, cost_incurred_date,
            # currency) because idle/warm pool capacity is tagged
            # DatabricksInstancePoolId, NOT ClusterId, so it never reaches
            # dbspend360_cloud_cost_explorer. The CP5 ClusterId-netting guard
            # already removed any pool cost that ALSO carried ClusterId, so
            # this is disjoint from the cluster/all-purpose/pipeline tabs.
            cloud_df = (
                spark.table(self.pool_cloud_cost_table)
                    .filter(
                        (F.col("cost_incurred_date") >= F.lit(start_dt)) &
                        (F.col("cost_incurred_date") <= F.lit(end_dt)) &
                        (F.col("instance_pool_id").isNotNull())
                    )
                    .select(
                        F.col("instance_pool_id"),
                        F.col("cost_incurred_date").alias("usage_date"),
                        F.col("currency"),
                        F.col("cloud_cost"),
                    )
                    # Native grain is already (pool, day, currency); the
                    # dropDuplicates is a defensive guard against accidental
                    # dupes that would fan the attach-join out.
                    .dropDuplicates(["instance_pool_id", "usage_date", "currency"])
            )

            dbu_empty = dbu_df.limit(1).count() == 0
            cloud_empty = cloud_df.limit(1).count() == 0
            # An idle-only window has pool cloud cost but ZERO DBU rows (idle
            # capacity emits no system.billing.usage rows, plan §4.1) — the
            # common case the pool tab exists for. So we must NOT short-circuit
            # on empty DBU alone; only skip when BOTH are empty.
            if dbu_empty and cloud_empty:
                self.logger.info(
                    "No pool DBU or cloud rows in this date window; nothing to roll up."
                )
                log_audit_run(
                    self.audit_table, self.TABLE_NAME, start_dt, end_dt,
                    "SUCCESS", 0, "No DBU or cloud data in window",
                )
                return

            # Flatten staging to the columns the rollup needs. Working off a
            # flat projection (rather than the aliased full-schema dbu_df)
            # keeps names unambiguous when we union synthesized overhead rows
            # and attach cloud below.
            dbu_proj = dbu_df.select(
                F.col("dbu.instance_pool_id").alias("instance_pool_id"),
                F.col("dbu.cluster_id").alias("cluster_id"),
                F.col("dbu.usage_date").alias("usage_date"),
                F.col("dbu.workspace_id").alias("workspace_id"),
                F.col("dbu.databricks_cost").alias("databricks_cost"),
                F.col("dbu.currency").alias("currency"),
                F.col("dbu.sku_name").alias("sku_name"),
            )

            # Collapse cloud to the overhead-carrier grain (pool, day). Pool VM
            # cost is pool-level, not attributable to a specific attached
            # cluster, so it lands on the synthetic __pool_overhead__ row (plan
            # §4.4). The rollup MERGE key has no currency, so exactly one
            # overhead row exists per pool-day; we sum across currency and warn
            # if a pool-day is ever multi-currency (then currency must join the
            # MERGE key, plan §4.4 note).
            cloud_agg = (
                cloud_df
                .groupBy("instance_pool_id", "usage_date")
                .agg(
                    F.sum("cloud_cost").alias("pool_cloud_cost"),
                    F.first("currency", ignorenulls=True).alias("cloud_currency"),
                    F.countDistinct("currency").alias("n_currencies"),
                )
            )
            multi_ccy = cloud_agg.filter(F.col("n_currencies") > 1)
            if multi_ccy.limit(1).count() > 0:
                self.logger.warning(
                    "POOL_CLOUD_MULTI_CURRENCY: %d (instance_pool_id, usage_date) "
                    "pairs carry >1 currency; cloud_cost is summed onto a single "
                    "overhead row because the MERGE key has no currency (plan §4.4). "
                    "Add currency to the key if this becomes routine.",
                    multi_ccy.count(),
                )
            cloud_agg = cloud_agg.select(
                "instance_pool_id", "usage_date", "pool_cloud_cost", "cloud_currency"
            )

            # Synthesize a __pool_overhead__ row for every pool-day that has
            # cloud cost but no existing overhead row (idle-only days, or
            # pool-days whose DBU never emitted the sentinel row). Driving the
            # synthesis from the cloud table — not LEFT-joining cloud onto the
            # rollup — is REQUIRED: idle pool-days have no rollup row at all, so
            # a rollup-driven join would silently drop the idle EC2 cost we most
            # want and break the §4.6 invariant (plan §4.4). Synthesized rows
            # carry databricks_cost = 0 and flow through the snapshot join below
            # so they pick up pool_name / node_type just like real rows.
            existing_overhead = (
                dbu_proj
                .filter(F.col("cluster_id") == "__pool_overhead__")
                .select("instance_pool_id", "usage_date")
                .distinct()
            )
            synth_overhead = (
                cloud_agg
                .join(existing_overhead, ["instance_pool_id", "usage_date"], "left_anti")
                .select(
                    F.col("instance_pool_id"),
                    F.lit("__pool_overhead__").alias("cluster_id"),
                    F.col("usage_date"),
                    F.lit(None).cast("string").alias("workspace_id"),
                    F.lit(0.0).cast("double").alias("databricks_cost"),
                    F.col("cloud_currency").alias("currency"),
                    F.lit(None).cast("string").alias("sku_name"),
                )
            )

            # Attach cloud onto overhead rows only; per-cluster rows keep
            # cloud_cost = NULL so the UI renders "—" with the pool-level note
            # (plan §5) and the pool-day SUM stays exact. Overhead rows on a
            # pool-day with no cloud also stay NULL (the §5 "unavailable" note).
            dbu_all = (
                dbu_proj.unionByName(synth_overhead)
                .join(cloud_agg, ["instance_pool_id", "usage_date"], "left")
                .withColumn(
                    "cloud_cost",
                    F.when(
                        F.col("cluster_id") == "__pool_overhead__",
                        F.col("pool_cloud_cost"),
                    ),
                )
                .drop("pool_cloud_cost", "cloud_currency")
            )

            pools_df = self._load_pool_snapshot().alias("p")

            # LEFT join so DBU/overhead rows with no matching snapshot row still
            # land in the target. The snapshot-missing path is signalled by
            # pool_name being NULL post-join; the COALESCE that paints the
            # fallback name comes AFTER pool_snapshot_missing is captured
            # (plan CP3 implementation notes).
            joined = (
                dbu_all.alias("dbu").join(
                    pools_df,
                    on=(F.col("dbu.instance_pool_id") == F.col("p.instance_pool_id")),
                    how="left",
                )
                .withColumn("pool_snapshot_missing", F.col("p.pool_name").isNull())
            )

            select_cols = [
                F.col("dbu.instance_pool_id").alias("instance_pool_id"),
                F.col("dbu.cluster_id").alias("cluster_id"),
                F.col("dbu.usage_date").alias("usage_date"),
                F.col("dbu.workspace_id").alias("workspace_id"),
                F.coalesce(
                    F.col("p.pool_name"),
                    F.concat(F.lit("Pool "), F.col("dbu.instance_pool_id")),
                ).alias("pool_name"),
                F.col("p.node_type").alias("node_type"),
                F.col("p.min_idle_instances").alias("min_idle_instances"),
                F.col("p.max_capacity").alias("max_capacity"),
                F.col("p.idle_instance_autotermination_minutes").alias(
                    "idle_instance_autotermination_minutes"
                ),
                F.col("pool_snapshot_missing"),
                F.col("p.pool_deleted_at").alias("pool_deleted_at"),
                F.col("dbu.databricks_cost").alias("databricks_cost"),
                # Pool EC2/EBS attached above (plan §4.4): real value on the
                # __pool_overhead__ row, NULL on per-cluster rows and on
                # overhead rows whose explorer cost hasn't landed (plan §5).
                # The total_cost COALESCE keeps the total safe either way.
                F.col("dbu.cloud_cost").alias("cloud_cost"),
                F.col("dbu.currency").alias("currency"),
                F.col("dbu.sku_name").alias("sku_name"),
            ]

            final_df = joined.select(*select_cols)

            final_df = (
                final_df
                .withColumn(
                    "total_cost",
                    F.coalesce(F.col("databricks_cost"), F.lit(0.0))
                    + F.coalesce(F.col("cloud_cost"), F.lit(0.0)),
                )
                .withColumn("created_at", F.current_timestamp())
                .withColumn("updated_at", F.current_timestamp())
            )
            final_df = safe_cache(final_df)

            row_count = final_df.count()

            validate_source_schema(
                final_df,
                {"instance_pool_id": "string", "cluster_id": "string",
                 "usage_date": "date", "databricks_cost": "double",
                 "cloud_cost": "double", "total_cost": "double",
                 "pool_snapshot_missing": "boolean"},
                self.target_table, self.logger,
            )
            validate_no_negative_costs(
                final_df,
                ["databricks_cost", "cloud_cost", "total_cost"],
                self.target_table, self.logger,
            )
            validate_currency_consistency(final_df, "currency", self.target_table, self.logger)

            # Guard (plan §4.4): the MERGE key is (instance_pool_id, cluster_id,
            # usage_date) with NO workspace_id, and pool cloud lands ONLY on the
            # synthetic __pool_overhead__ row. instance_pool_id is workspace-
            # scoped in practice, so exactly one overhead row exists per
            # (pool, day). If a pool ever went cross-workspace and emitted >1
            # overhead row on a single day, the cloud_agg left-join would attach
            # pool_cloud_cost to BOTH -> _assert_reconciliation would see 2x and
            # fail, and the MERGE would error with "multiple source rows
            # matched". Fail fast here with an actionable message (add
            # workspace_id to the key if pools go cross-workspace) instead.
            overhead_dupes = (
                final_df
                .filter(F.col("cluster_id") == "__pool_overhead__")
                .groupBy("instance_pool_id", "usage_date")
                .count()
                .filter(F.col("count") > 1)
            )
            if overhead_dupes.limit(1).count() > 0:
                dupe_count = overhead_dupes.count()
                sample = overhead_dupes.limit(5).collect()
                sample_str = "; ".join(
                    f"(pool={r['instance_pool_id']}, date={r['usage_date']}, rows={r['count']})"
                    for r in sample
                )
                raise DataQualityError(
                    f"Pool overhead-row uniqueness violated: {dupe_count} "
                    f"(instance_pool_id, usage_date) keys carry >1 __pool_overhead__ "
                    f"row (suspected cross-workspace pool). The MERGE key has no "
                    f"workspace_id, so cloud would double-attach, reconciliation would "
                    f"see 2x, and the MERGE would fail with 'multiple source rows "
                    f"matched'. Add workspace_id to the overhead/MERGE key if pools go "
                    f"cross-workspace. Sample: {sample_str}"
                )

            # §4.6 hard gate on the pre-MERGE source (the MERGE writes these
            # values verbatim for the window). Aborts before any write if the
            # rollup's pool cloud_cost diverges from the explorer.
            self._assert_reconciliation(final_df, cloud_df, start_dt, end_dt)

            target = DeltaTable.forName(spark, self.target_table)
            (target.alias("t")
                .merge(
                    final_df.alias("s"),
                    "t.instance_pool_id = s.instance_pool_id "
                    "AND t.cluster_id = s.cluster_id "
                    "AND t.usage_date = s.usage_date",
                )
                .whenMatchedUpdate(set={
                    "workspace_id": "s.workspace_id",
                    "pool_name": "s.pool_name",
                    "node_type": "s.node_type",
                    "min_idle_instances": "s.min_idle_instances",
                    "max_capacity": "s.max_capacity",
                    "idle_instance_autotermination_minutes":
                        "s.idle_instance_autotermination_minutes",
                    "pool_snapshot_missing": "s.pool_snapshot_missing",
                    "pool_deleted_at": "s.pool_deleted_at",
                    "databricks_cost": "s.databricks_cost",
                    "cloud_cost": "s.cloud_cost",
                    "total_cost": "s.total_cost",
                    "currency": "s.currency",
                    "sku_name": "s.sku_name",
                    "updated_at": "current_timestamp()",
                })
                .whenNotMatchedInsert(values={
                    "instance_pool_id": "s.instance_pool_id",
                    "cluster_id": "s.cluster_id",
                    "usage_date": "s.usage_date",
                    "workspace_id": "s.workspace_id",
                    "pool_name": "s.pool_name",
                    "node_type": "s.node_type",
                    "min_idle_instances": "s.min_idle_instances",
                    "max_capacity": "s.max_capacity",
                    "idle_instance_autotermination_minutes":
                        "s.idle_instance_autotermination_minutes",
                    "pool_snapshot_missing": "s.pool_snapshot_missing",
                    "pool_deleted_at": "s.pool_deleted_at",
                    "databricks_cost": "s.databricks_cost",
                    "cloud_cost": "s.cloud_cost",
                    "total_cost": "s.total_cost",
                    "currency": "s.currency",
                    "sku_name": "s.sku_name",
                    "created_at": "current_timestamp()",
                    "updated_at": "current_timestamp()",
                })
                .execute()
            )

            # §4.6 advisory monitor (mirrors the explorer); run while final_df
            # is still cached, before we release it.
            self._monitor_post_write(final_df, start_dt, end_dt)

            safe_unpersist(final_df)
            get_merge_metrics(self.target_table, self.logger)

            validate_post_merge(
                self.target_table, "usage_date",
                start_dt, end_dt, row_count, self.logger,
            )

            # Reconciliation already asserted on the pre-MERGE source (§4.6);
            # cloud_cost is now populated from dbspend360_pool_cloud_cost_explorer.
            log_audit_run(
                self.audit_table, self.TABLE_NAME, start_dt, end_dt,
                "SUCCESS", row_count, "",
            )
            self.logger.info(
                f"Merged {row_count} rows into {self.target_table} "
                f"for {start_dt} → {end_dt}."
            )

        except Exception as e:
            msg = str(e)[:1000]
            self.logger.error(f"Run failed: {msg}")
            try:
                log_audit_run(
                    self.audit_table, self.TABLE_NAME, start_dt, end_dt, "FAILED", 0, msg,
                )
            except Exception:
                self.logger.error("Failed to write FAILED audit entry")
            raise

In [ ]:
# =======================================================
# APP
# =======================================================
class TotalPoolSpendsApp:

    def __init__(self):
        catalog = dbutils.widgets.get("catalog")
        schema = dbutils.widgets.get("schema")
        ov_days = get_overlap_days(dbutils.widgets.get("overlap_days"), logger=logger)

        self.client = TotalPoolSpendsClient(
            audit_table=build_table_fqn(catalog, schema, "dbspend360_audit_log"),
            pool_cloud_cost_table=build_table_fqn(catalog, schema, "dbspend360_pool_cloud_cost_explorer"),
            databricks_cost_table=build_table_fqn(catalog, schema, "dbspend360_pool_dbu_cost"),
            target_table=build_table_fqn(catalog, schema, "dbspend360_total_pool_spends"),
            error_log_table=build_table_fqn(catalog, schema, "dbspend360_error_log"),
            overlap_days=ov_days,
            logger=logger,
        )

    def run(self):
        self.client.build_total_pool_spends()

In [ ]:
# =======================================================
# Execute
# =======================================================
app = TotalPoolSpendsApp()
app.run()